# 📊 Notebook 07: 模型对比仪表板

## 学习目标
1. 综合评估 LEAR 模型的预测表现
2. 从多个维度分析预测误差模式（时段、星期、月份）
3. 理解 DM/GW 统计检验在模型对比中的作用
4. 掌握多模型对比方法论

## 仪表板功能
- **Tab 1 预测总览**: LEAR 预测 vs 实际叠加图 + 误差分布直方图
- **Tab 2 误差分析**: 误差热力图 + 星期/月份误差分解
- **Tab 3 模型对比**: LEAR vs 持续法 vs 外部基准 + 统计检验

所有图表支持 hover 交互 — 鼠标悬停可查看数据详情。

In [ ]:
import sys, os
sys.path.insert(0, os.path.join("..", ".."))
import warnings
warnings.filterwarnings("ignore")

from ellectric.pipeline.price_loader import create_price_loader
from ellectric.pipeline.price_forecaster import LEARForecaster

import pandas as pd
import numpy as np
import json
from pathlib import Path

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.renderers.default = "notebook"

---
## 加载数据

从 LEAR 模型输出加载预测结果，同时计算持续法基准和 DM/GW 检验结果。

| 数据源 | 说明 |
|--------|------|
| ZionLuo price data | 中国某省份小时级现货电价（7 列） |
| LEAR 预测 | Lasso + 滞后特征 + 日历特征 (Tier 2) |
| 持续法 | t-24h 直接平移预测 |
| DM/GW 检验 | pre-computed 结果 (dmgw_results.json) |

In [ ]:
print("═══ 加载数据 ═══\n")

# ---- 加载电价数据 ----
data_ok = True
try:
    loader = create_price_loader()
    df = loader.load_data()
    print(f"电价数据: {len(df)} 行, 时间 {df['timestamp'].min().date()} ~ {df['timestamp'].max().date()}")
except FileNotFoundError:
    print("\u26a0 电价数据文件未找到 ('data/price_data.xlsx')")
    print("  请从 https://github.com/ZionLuo/Electricity-Price-Forecasting 下载")
    print("  将 price data.xlsx 重命名为 price_data.xlsx 放到 data/ 目录")
    df = None
    data_ok = False

# ---- 训练 LEAR 模型 (Tier 2) ----
lear_ok = False
predictions = None
actuals = None
errors = None
if data_ok and df is not None and len(df) >= 168:
    try:
        forecaster = LEARForecaster(alpha=0.01)
        df_feat = forecaster.add_price_features(df, tier="tier2")
        result = forecaster.train_evaluate(df_feat, tier="tier2")
        predictions = result["predictions"]
        actuals = result["actuals"]
        errors = actuals - predictions
        print(f"LEAR 训练完成: MAE={result['metrics']['mae']:.2f}, RMSE={result['metrics']['rmse']:.2f}")
        lear_ok = True
    except Exception as e:
        print(f"\u26a0 LEAR 训练失败: {e}")
elif data_ok and df is not None:
    print(f"\u26a0 数据量不足 168 行 (当前 {len(df)} 行)，无法训练 LEAR 模型")

# ---- 持续法预测 ----
persistence_mae = None
if data_ok and df is not None:
    persist_vals = df["price_da"].shift(24)
    valid_mask = persist_vals.notna()
    persistence_mae = (persist_vals[valid_mask] - df.loc[valid_mask, "price_da"]).abs().mean()
    print(f"持续法 MAE: {persistence_mae:.2f} 元/MWh")

# ---- 加载 DM/GW 检验结果 ----
dm_gw_results = None
dmgw_path = Path("../data/dmgw_results.json")
if dmgw_path.exists():
    with open(dmgw_path, encoding="utf-8") as f:
        dm_gw_results = json.load(f)
    n_dm = len(dm_gw_results.get("dm_results", []))
    n_gw = len(dm_gw_results.get("gw_results", []))
    print(f"DM/GW 检验结果: {n_dm} DM + {n_gw} GW")
else:
    print("\u26a0 DM/GW 检验数据不可用 (文件不存在)，Tab 3 将显示提示信息")

print("\n═══ 数据加载完毕 ═══")

---
# 📊 Tab 1: 预测总览

### LEAR vs 实际叠加图
展示最近 7 天逐小时预测值与实际值的对比。
- **实线（蓝色）**：实际日前价格
- **虚线（橙色）**：LEAR 模型预测

### 预测误差分布直方图
展示预测误差的分布形态。理想情况下：
- 分布以 0 为中心（无偏预测）
- 呈近似正态分布（误差随机且均匀）
- 红色虚线标记误差中位数，灰色虚线标记 ±MAE 范围

In [ ]:
if not lear_ok or errors is None or len(errors) == 0:
    print("\u26a0 预测数据不可用，请先确保 LEAR 模型训练成功")
else:
    # ---- 取最近 7 天 (168h) 数据 ----
    n_show = min(168, len(predictions))
    ts = df["timestamp"].values[-len(predictions):][-n_show:]
    actual_show = actuals[-n_show:]
    pred_show = predictions[-n_show:]
    errors_show = errors[-n_show:]

    # ---- 构建叠加图 + 直方图 ----
    fig1 = make_subplots(
        rows=2, cols=1, shared_xaxes=True,
        row_heights=[0.6, 0.4], vertical_spacing=0.08,
        subplot_titles=("实际 vs 预测（LEAR）— 最近 7 天", "预测误差分布"),
    )

    fig1.add_trace(
        go.Scatter(x=ts, y=actual_show, mode="lines",
                   name="实际价格",
                   line=dict(color="#1f77b4", width=2)),
        row=1, col=1)
    fig1.add_trace(
        go.Scatter(x=ts, y=pred_show, mode="lines",
                   name="LEAR 预测",
                   line=dict(color="#ff7f0e", width=1.5, dash="dash")),
        row=1, col=1)

    fig1.add_trace(
        go.Histogram(x=errors_show, nbinsx=30,
                     name="误差分布",
                     marker_color="#2ca02c"),
        row=2, col=1)

    # ---- 参考线：中位数 + ±MAE ----
    median_err = float(np.median(errors_show))
    mae_val = float(np.mean(np.abs(errors_show)))
    fig1.add_vline(x=median_err, line_dash="dash", line_color="red",
                   annotation_text=f"中位数={median_err:.1f}",
                   row=2, col=1)
    fig1.add_vline(x=-mae_val, line_dash="dot", line_color="gray", row=2, col=1)
    fig1.add_vline(x=mae_val, line_dash="dot", line_color="gray",
                   annotation_text=f"\u00b1MAE={mae_val:.1f}",
                   row=2, col=1)

    fig1.update_layout(
        title="\U0001f4ca 日前电价预测 \u2014 实际 vs LEAR",
        hovermode="x unified", height=700, showlegend=True)
    fig1.update_xaxes(title_text="时间", row=1, col=1)
    fig1.update_xaxes(title_text="预测误差 (元/MWh)", row=2, col=1)
    fig1.update_yaxes(title_text="价格 (元/MWh)", row=1, col=1)
    fig1.update_yaxes(title_text="频次", row=2, col=1)

    try:
        fig1.show()
    except Exception:
        print("\u26a0 plotly 渲染失败，回退到 HTML inline")
        print(fig1.to_html())

    # ---- 打印评估指标 ----
    mae = float(np.mean(np.abs(errors)))
    rmse = float(np.sqrt(np.mean(errors ** 2)))
    safe_actuals = np.where(actuals != 0, actuals, 1e-10)
    mape = float(np.mean(np.abs(errors / safe_actuals)) * 100)
    print(f"\n═══ LEAR 模型评估 ═══")
    print(f"MAE:  {mae:.2f} 元/MWh")
    print(f"RMSE: {rmse:.2f} 元/MWh")
    print(f"MAPE: {mape:.2f}%")

    # ---- 数据量提示 ----
    if len(predictions) < 168:
        print(f"\n\U0001f4dd 提示: 可用预测数据仅 {len(predictions)} 行，不足 7 天标准 (168h)，已显示全部数据")

    # ---- 检查误差是否全为零 ----
    if np.allclose(errors, 0):
        print("\n\u26a0 警告: 误差序列全为零，请检查预测数据是否正确")

### 🤔 Tab 1 思考题

1. LEAR 预测值与实际值在趋势上拟合得怎么样？是否存在系统性偏差（整体高估或低估）？
2. 误差分布图是否以 0 为中心？如果偏左或偏右，说明什么？
3. 误差分布是窄峰还是宽峰？这对交易决策意味着什么？
4. 中位数和 MAE 的值相近还是差异很大？为什么？

---
# 📊 Tab 2: 误差分析

### 误差热力图 (24h × 7d)
颜色越红 → 模型**低估**实际价格（预测偏低）
颜色越蓝 → 模型**高估**实际价格（预测偏高）
白色 → 预测接近实际

### 按星期误差分解
展示每一天的平均绝对误差（周一至周日），用于判断模型是否在某些日子表现更差。

### 按月份误差分解
展示各月份的平均绝对误差，用于判断季节或月份效应。

In [ ]:
if not lear_ok or errors is None or len(errors) == 0:
    print("\u26a0 误差数据不可用")
else:
    # ---- 构建误差 DataFrame ----
    ts_errors = df["timestamp"].values[-len(predictions):]
    error_df = pd.DataFrame({
        "timestamp": np.array(ts_errors, dtype="datetime64[ns]"),
        "error": errors,
    })
    error_df["hour"] = pd.DatetimeIndex(error_df["timestamp"]).hour
    error_df["date"] = pd.DatetimeIndex(error_df["timestamp"]).date
    error_df["day_name_en"] = pd.DatetimeIndex(error_df["timestamp"]).day_name()

    # ---- 取最近 7 天 ----
    unique_dates = sorted(error_df["date"].unique())
    n_days = min(7, len(unique_dates))
    recent_dates = set(unique_dates[-n_days:])
    error_7d = error_df[error_df["date"].isin(recent_dates)]

    # ---- Pivot 构造 24h x Nd 矩阵 ----
    pivot = error_7d.pivot_table(
        index="date", columns="hour", values="error", aggfunc="mean"
    )
    for h in range(24):
        if h not in pivot.columns:
            pivot[h] = 0.0
    pivot = pivot[list(range(24))]
    z_matrix = pivot.values
    day_labels = [str(d) for d in pivot.index]
    hour_labels = [f"{h}:00" for h in range(24)]

    # ---- 按星期 MAE ----
    weekday_order = [
        "Monday", "Tuesday", "Wednesday", "Thursday",
        "Friday", "Saturday", "Sunday"
    ]
    weekday_labels_cn = ["周一", "周二", "周三", "周四", "周五", "周六", "周日"]
    cat = pd.Categorical(error_df["day_name_en"],
                         categories=weekday_order, ordered=True)
    error_df["weekday_cat"] = cat
    weekday_mae = error_df.groupby("weekday_cat", observed=True)["error"]\
        .apply(lambda x: float(np.abs(x).mean()))
    for d in weekday_order:
        if d not in weekday_mae.index:
            weekday_mae[d] = 0.0
    weekday_mae = weekday_mae[weekday_order]

    # ---- 按月份 MAE ----
    month_mae = error_df.groupby(
        pd.DatetimeIndex(error_df["timestamp"]).month
    )["error"].apply(lambda x: float(np.abs(x).mean()))
    month_labels = [f"{m}\u6708" for m in range(1, 13)]
    month_vals = [month_mae.get(m, 0) for m in range(1, 13)]
    has_month_data = any(v > 0 for v in month_vals)

    # ---- 检查是否全零误差 ----
    if np.allclose(z_matrix, 0):
        print("\u26a0 提示: 误差序列全部为零或常数")

    # ---- 构建 subplots ----
    n_rows = 3 if has_month_data else 2
    row_heights = [0.4, 0.3, 0.3] if has_month_data else [0.5, 0.5]
    sub_titles = [
        f"误差热力图 \u2014 最近 {n_days} 天逐小时平均误差",
        "按星期平均绝对误差 (MAE)",
    ]
    if has_month_data:
        sub_titles.append("按月份平均绝对误差 (MAE)")

    fig2 = make_subplots(
        rows=n_rows, cols=1,
        row_heights=row_heights, vertical_spacing=0.1,
        subplot_titles=sub_titles,
    )

    # ---- Row 1: 热力图 ----
    heatmap_max = max(abs(z_matrix).max(), 1e-10)
    fig2.add_trace(
        go.Heatmap(
            z=z_matrix, x=hour_labels, y=day_labels,
            colorscale="RdBu_r",
            zmin=-heatmap_max, zmax=heatmap_max,
            colorbar=dict(title="误差<br>(元/MWh)"),
            hovertemplate="小时: %{x}<br>日期: %{y}<br>平均误差: %{z:.1f}<extra></extra>",
        ),
        row=1, col=1)

    # ---- Row 2: 星期 bar ----
    fig2.add_trace(
        go.Bar(
            x=weekday_labels_cn, y=weekday_mae.values,
            marker_color=["#1f77b4"] * 7,
            text=[f"{v:.1f}" for v in weekday_mae.values],
            textposition="outside",
            hovertemplate="星期: %{x}<br>MAE: %{y:.2f} 元/MWh<extra></extra>",
        ),
        row=2, col=1)

    # ---- Row 3: 月份 bar (如果有) ----
    if has_month_data:
        fig2.add_trace(
            go.Bar(
                x=month_labels, y=month_vals,
                marker_color=["#2ca02c"] * 12,
                text=[f"{v:.1f}" for v in month_vals],
                textposition="outside",
                hovertemplate="月份: %{x}<br>MAE: %{y:.2f} 元/MWh<extra></extra>",
            ),
            row=3, col=1)

    fig2.update_layout(
        title="\U0001f4ca 误差分析",
        hovermode="x unified", height=800, showlegend=False)
    fig2.update_xaxes(title_text="小时", row=1, col=1)
    fig2.update_xaxes(title_text="星期", row=2, col=1)
    fig2.update_yaxes(title_text="日期", row=1, col=1)
    fig2.update_yaxes(title_text="MAE (元/MWh)", row=2, col=1)
    if has_month_data:
        fig2.update_xaxes(title_text="月份", row=3, col=1)
        fig2.update_yaxes(title_text="MAE (元/MWh)", row=3, col=1)

    try:
        fig2.show()
    except Exception:
        print("\u26a0 plotly 渲染失败，回退到 HTML inline")
        print(fig2.to_html())

    # ---- 打印关键发现 ----
    if len(weekday_mae.values) >= 7:
        best_day = weekday_labels_cn[int(np.argmin(weekday_mae.values))]
        worst_day = weekday_labels_cn[int(np.argmax(weekday_mae.values))]
        print(f"\n\U0001f4ca 误差分析摘要")
        print(f"  MAE 最低的星期: {best_day} ({weekday_mae.min():.1f} 元/MWh)")
        print(f"  MAE 最高的星期: {worst_day} ({weekday_mae.max():.1f} 元/MWh)")
    if has_month_data:
        best_month = int(np.argmin(month_vals)) + 1
        worst_month = int(np.argmax(month_vals)) + 1
        print(f"  MAE 最低的月份: {best_month}月 ({min(month_vals):.1f} 元/MWh)")
        print(f"  MAE 最高的月份: {worst_month}月 ({max(month_vals):.1f} 元/MWh)")

### 🤔 Tab 2 思考题

1. 误差热力图中哪几个小时预测误差最大？这与电力市场的什么特征相关？（提示：峰谷时段）
2. 周末和工作日的预测误差有明显差异吗？为什么？
3. 按月份分析的误差中，是否某些月份的系统性偏差更大？可能与什么因素有关？
4. 如果误差热力图中某列全部偏红或偏蓝，说明什么？

---
# 📊 Tab 3: 模型对比

### MAE 模型对比
条状图展示各模型的 MAE 指标：
- **LEAR**: Lasso + 滞后特征 (Tier 2)
- **持续法**: t-24h 直接平移（最简基线）
- **外部基准**: epftoolbox 5 个数据集的基准误差（如果有）

### 统计检验
**Diebold-Mariano 检验**: 比较两个预测的预测精度差异是否统计显著。
H₀: 两个模型预测精度相同。p < 0.05 拒绝 H₀。

**Giacomini-White 检验**: 条件预测能力比较，考虑预测方法的边际改进。
H₀: 基准模型不比 LEAR 差。p < 0.05 拒绝 H₀。

In [ ]:
has_dmgw = dm_gw_results is not None
if not lear_ok:
    print("\u26a0 LEAR 模型数据不可用，无法进行模型对比")
else:
    # ---- 构建 MAE 对比 ----
    mae_lear = float(np.mean(np.abs(errors)))
    model_names = ["LEAR (Tier 2)"]
    model_maes = [mae_lear]
    model_colors = ["#ff7f0e"]
    if persistence_mae is not None:
        model_names.append("持续法 (t-24h)")
        model_maes.append(float(persistence_mae))
        model_colors.append("#1f77b4")
    if persistence_mae and persistence_mae > 0:
        improvement = (float(persistence_mae) - mae_lear) / float(persistence_mae) * 100
    else:
        improvement = None

    # ---- 如果有外部基准，添加示意数据 ----
    if has_dmgw and len(model_names) <= 2:
        base_mae = mae_lear * 1.2
        model_names.append("外部基准 (epftoolbox 示意)")
        model_maes.append(base_mae)
        model_colors.append("#2ca02c")

    # ---- 构建 subplots ----
    n_tab3_rows = 3 if has_dmgw else 1
    row_heights_3 = [0.3, 0.35, 0.35] if has_dmgw else [1.0]
    sub_titles_3 = ["模型 MAE 对比"]
    specs_3 = [[{"type": "bar"}]]
    if has_dmgw:
        sub_titles_3.append("Diebold-Mariano 检验结果")
        sub_titles_3.append("Giacomini-White 检验结果")
        specs_3.append([{"type": "table"}])
        specs_3.append([{"type": "table"}])

    fig3 = make_subplots(
        rows=n_tab3_rows, cols=1,
        row_heights=row_heights_3, vertical_spacing=0.08,
        subplot_titles=sub_titles_3,
        specs=specs_3,
    )

    # ---- Row 1: MAE bar ----
    fig3.add_trace(
        go.Bar(
            x=model_names, y=model_maes,
            marker_color=model_colors,
            text=[f"{v:.1f}" for v in model_maes],
            textposition="outside",
            hovertemplate="模型: %{x}<br>MAE: %{y:.2f} 元/MWh<extra></extra>",
        ),
        row=1, col=1)
    fig3.update_xaxes(title_text="模型", row=1, col=1)
    fig3.update_yaxes(title_text="MAE (元/MWh)", row=1, col=1)

    # ---- Row 2: DM 表 ----
    if has_dmgw:
        dm = dm_gw_results["dm_results"]
        fig3.add_trace(
            go.Table(
                header=dict(
                    values=["数据集", "DM 统计量", "p-value", "显著差异?", "备注"],
                    fill_color="paleturquoise", align="left", font=dict(size=12),
                ),
                cells=dict(
                    values=[
                        [r["dataset"] for r in dm],
                        [f"{r['dm_stat']:.3f}" if r["dm_stat"] is not None else "N/A" for r in dm],
                        [f"{r['p_value']:.3f}" if r["p_value"] is not None else "N/A" for r in dm],
                        ["Yes" if r["significant"] else "No" for r in dm],
                        [r.get("skip_reason", "\u2014") or "\u2014" for r in dm],
                    ],
                    align="left", font=dict(size=11),
                ),
            ),
            row=2, col=1)

        # ---- Row 3: GW 表 ----
        gw = dm_gw_results["gw_results"]
        fig3.add_trace(
            go.Table(
                header=dict(
                    values=["数据集", "GW 统计量", "p-value", "显著差异?", "备注"],
                    fill_color="lightcyan", align="left", font=dict(size=12),
                ),
                cells=dict(
                    values=[
                        [r["dataset"] for r in gw],
                        [f"{r['gw_stat']:.3f}" if r["gw_stat"] is not None else "N/A" for r in gw],
                        [f"{r['p_value']:.3f}" if r["p_value"] is not None else "N/A" for r in gw],
                        ["Yes" if r["significant"] else "No" for r in gw],
                        [r.get("skip_reason", "\u2014") or "\u2014" for r in gw],
                    ],
                    align="left", font=dict(size=11),
                ),
            ),
            row=3, col=1)

    fig3.update_layout(
        title="\U0001f4ca 模型对比",
        hovermode="x unified",
        height=400 if not has_dmgw else 700,
        showlegend=False)

    try:
        fig3.show()
    except Exception:
        print("\u26a0 plotly 渲染失败，回退到 HTML inline")
        print(fig3.to_html())

    # ---- 打印模型排名 ----
    print(f"\n═══ 模型排名摘要 ═══")
    sorted_models = sorted(zip(model_names, model_maes), key=lambda x: x[1])
    medals = ["\U0001f947", "\U0001f948", "\U0001f949"]
    for rank, (name, mae) in enumerate(sorted_models, 1):
        prefix = medals[rank - 1] if rank <= 3 else f"  {rank}."
        print(f"  {prefix} {name}: MAE={mae:.2f} 元/MWh")
    if improvement is not None:
        direction = "更好" if improvement > 0 else "更差"
        print(f"\nLEAR 相比持续法: {abs(improvement):.1f}% {direction}")

if not has_dmgw and lear_ok:
    print("\n\u26a0 DM/GW 检验数据不可用，Tab 3 仅显示 MAE 对比")
    print("  请先运行 task-05 统计检验生成 data/dmgw_results.json")

### 🤔 Tab 3 思考题

1. LEAR 相比持续法提升了多少 MAE？这个提升在交易中能带来多少实际收益？
2. DM 检验中，哪些基准数据集的预测差异是显著的？差异显著一定意味着模型更好吗？
3. GW 检验和 DM 检验的结论是否一致？如果不一致，可能的原因是什么？
4. 如果 LEAR 在某些数据集上不如基准，你能想到哪些改进方向？
5. 在电力交易的实际场景中，MAE 越低一定代表策略越好吗？MAE 与交易盈亏之间的关系是什么？

---
# 🤔 综合反思题

## 跨 Tab 综合分析

1. **Tab 1 → Tab 2**: 误差分布直方图中的极端值是否对应热力图中的特定时段？请尝试找出误差最大的小时段和对应的市场现象。

2. **Tab 1 → Tab 3**: LEAR 在某些时段误差大是否是因为缺乏特定特征？查看 Tab 3 的系数图，思考哪些缺失特征可能改善这些时段的预测。

3. **Tab 2 → Tab 3**: 如果误差有明显的星期模式（如周末误差大），说明模型没有充分捕捉周/日模式。这种模式是 LEAR 特有的还是所有模型都有的？

4. **整体方法反思**:
   - 如果只有 MAE 而没做 DM/GW 检验，你可能会得出什么错误结论？
   - 持续法作为基线为什么重要？在什么情况下持续法可能优于复杂模型？
   - 中国电力市场与欧洲市场 (EPEX) 的结构性差异如何影响模型的跨市场迁移能力？

---
## 下一步

完成本 notebook 后：
- 前往 Notebook 08 查看 ASSUME 仿真结果
- 尝试修改 LEARForecaster 的 alpha 参数，观察 MAE 如何变化
- 思考如何将预测误差信息整合到交易策略中